# 第8周 · 周四：实验平台架构——实验从"手工"到"工程化"

> 难度：⭐⭐⭐⭐ | 定位：理解实验平台的技术架构，掌握分析师在平台生态中的角色
> 前置依赖：W7-W8所有实验知识 | 后续用途：实验平台选型、指标口径对齐、平台问题排查
> 面试地位：高级分析师必考——"你用过什么实验平台？你觉得它有什么问题？"
> 配对课程：01python_learn W8D4（自定义Transformer）

---

## 📘 区块1：核心概念与第一性原理

### 1.1 实验平台的本质——"让实验规模化"

一个实验平台的核心价值不是"帮你做统计检验"——任何一个分析师用Excel都能做t检验。实验平台的核心价值是**让实验规模化、标准化、可审计**。

没有平台时：
- 工程师手动写分流逻辑→容易出错
- 每个分析师自己算指标→口径不一致
- 实验结果散落在各种文档里→无法追溯历史
- 同一个用户可能同时被多个实验命中→互相干扰

有了平台后：
- 统一的分流服务→所有实验共用同一套hash
- 统一的指标定义→"GMV"在整个公司口径一致
- 统一的实验报表→所有实验可对比、可追溯
- 实验间互斥/正交管理→避免多实验互相干扰

### 1.2 实验平台的"五脏六腑"——六大核心模块

```
┌─────────────────────────────────────────┐
│        ① 实验配置（UI/API）              │  ← 产品/运营配置实验参数
├─────────────────────────────────────────┤
│        ② 分流服务（SDK/Server）          │  ← hash(user_id+layer_id)→分配到哪组
├─────────────────────────────────────────┤
│        ③ 数据采集（埋点/日志）           │  ← 记录每个用户的分组和所有行为
├─────────────────────────────────────────┤
│        ④ 指标计算（ETL Pipeline）        │  ← 每天/每小时计算实验指标
├─────────────────────────────────────────┤
│        ⑤ 统计分析（计算引擎）            │  ← t检验/CUPED/序贯检验/多重检验
├─────────────────────────────────────────┤
│        ⑥ 结果展示（报表/看板）           │  ← 实验报告、趋势图、决策建议
└─────────────────────────────────────────┘
```

### 1.3 分析师需要关心的——不只是"看报表"

很多人以为分析师在实验平台中的角色就是"打开看板，看p值"。实际上，一个合格的分析师需要理解：

1. **分流算法**：平台用的是什么hash函数？分层分流和独立分流有什么区别？为什么这影响实验之间的独立性？
2. **指标口径**：平台上的"GMV"是怎么定义的？包含退款吗？包含运费吗？跟你看板上的口径一致吗？
3. **统计方法**：平台默认用了什么检验？有没有做CUPED？有没有做多重检验校正？p值是怎么算的？
4. **数据时间窗口**：平台的指标更新频率是多少？（T+1还是实时？）——这决定了你什么时候能看到结果

### 1.4 分流服务——实验平台的心脏

分流是实验平台最核心、最容易出错的部分。核心要求：
- **一致性**：同一个用户每次请求都分到同一组（除非实验结束）
- **均匀性**：两组用户的特征分布应该相同（可检验→SRM检查）
- **独立性**：不同实验之间的分流不应该互相影响（分层分流）

**常见分流算法**：
- **Hash分流**：hash(user_id + layer_id + experiment_id) % 100 → 0-49=对照组, 50-99=实验组
- **随机分流**：为每个用户生成一个0-1的随机数→存入用户属性表→按阈值分组

**Hash分流 vs 随机分流**：
- Hash：确定性的，不需要查表，但同一用户在不同层可能被分到不同组
- 随机：需要查表，但可以保证"同一用户在所有实验中都在同一组"（如果需要的话）

### 1.5 分层分流——让多个实验不互相干扰

大型平台同时运行数百个实验。如果每个实验独立分流，同一个用户可能被分到A实验的实验组和B实验的对照组——这个用户的体验是"混乱的"。

**分层（Layer）的解决方案**：
- 把流量分成多层（如UI层、算法层、内容层）
- 每层独立分流，保证同一用户在不同层被分到不同组是"均匀的"
- 同一层的实验互斥（一个用户只能在同一层的一个实验中）
- 不同层的实验正交（互相不影响）

**分层示例**：
```
Layer 1 (UI):     用户被分到"按钮颜色实验"的实验组
Layer 2 (算法):   同一用户被分到"推荐算法实验"的对照组
Layer 3 (内容):   同一用户被分到"内容排序实验"的实验组
```
这个用户在不同层的分组是独立的——一个层的分组不影响另一个层。

### 1.6 指标计算——"口径统一"的工程实现

这是分析师最容易和平台产生分歧的地方。平台的指标计算涉及：
- **数据源**：用的是实时流还是T+1批处理？
- **时间窗口**：指标统计的是"实验期间"还是"用户生命周期"？
- **去重逻辑**：同一个事件多次上报怎么处理？
- **异常值处理**：是否去掉了极端值（如单笔100万的订单）？
- **分母定义**：留存率的分母是"所有进入实验的用户"还是"实验期间活跃的用户"？

当平台显示的p=0.03，你自己算的是p=0.08——这0.05的差距大概率来自指标口径差异，而不是统计方法差异。

### 1.7 商业实验平台 vs 自建平台

| 维度 | 商业平台（Optimizely/Statsig/Eppo） | 自建平台 |
|------|-----------------------------------|---------|
| 开发成本 | 按流量付费，年费几十万到几百万 | 初期投入大，需要2-3个工程师×6个月 |
| 灵活性 | 有限——只能配置平台支持的功能 | 无限——任何分析都可以做 |
| 维护成本 | 低——供应商负责 | 高——需要持续投入 |
| 定制化 | 有限 | 无限 |
| 适用规模 | 中小公司、快速启动 | 大厂、有特殊需求 |

### 1.8 面试高频考点

**面试官常问**：
- "你们公司的实验平台是怎么设计的？"
- "你觉得现在的实验平台有什么可以改进的地方？"
- "分层分流是什么？为什么需要？"
- "平台的指标口径和你的分析口径不一致怎么办？"

**回答要点**：
- 描述六大模块：配置→分流→采集→指标→统计→展示
- 改进点：CUPED支持、序贯检验、非标准实验（Cluster/Switchback）
- 分层=按功能域隔离实验，层内互斥、层间正交
- 不一致时先查口径差异（数据源/窗口/去重/分母），再查统计方法差异</cell>


## 🧩 区块2：知识点详细拆解

### 2.1 Hash分流的数学原理与均匀性

**为什么重要**：理解hash算法是排查分流bug的基础。

一个好的hash函数需要满足：
- **均匀分布**：hash值在0到2^32-1之间均匀分布→分流比例精准
- **雪崩效应**：输入微小变化→输出巨大变化→相似用户不会被分到同一组
- **确定性**：同一输入永远输出同一hash→同一用户每次分到同一组

### 2.2 SRM（样本比例不匹配）检查——平台第一道防线

**为什么重要**：SRM是分流bug的最早信号。实验平台应该自动做SRM检查。

SRM检验：实际两组人数比例是否显著偏离设计的50:50？
- 如果P=0.04 → 偏离显著 → 分流有bug
- 常见原因：hash碰撞、设备ID不稳定、实验配置错误

### 2.3 指标口径管理——"同一个GMV，三个结果"

**为什么重要**：口径不一致是分析师和平台"打架"的首要原因。

三种常见的GMV口径：
- **订单级GMV**：用户下单后就计入（含未支付/退款）
- **支付级GMV**：用户支付后计入（含后续退款）
- **实收GMV**：扣除退款后的最终收入

不同平台/看板可能使用不同口径——分析师需要知道每个指标的定义。

### 2.4 统计引擎——平台帮你算了什么

现代实验平台通常内置：
- t检验（默认）、Welch t检验（方差不齐）
- CUPED（方差缩减）—需要实验前数据
- 序贯检验（允许中间分析但控制α）
- 多重检验校正（Bonferroni/FDR）
- Bootstrap CI（非参数置信区间）
- 分层分析（如分iOS/Android看）

### 2.5 实验间依赖管理——层与域的设计

**为什么重要**：同一用户被多个实验命中时，可能出现意想不到的交互效应。

**层（Layer）设计原则**：
- 同一层的实验互斥（一个用户只能在一个实验中）
- 不同层的实验正交（独立分流）
- 争议点：如果UI层的改版影响了用户对算法层改版的感知→"正交"只是近似的

### 2.6 实时数据 vs T+1数据

**为什么重要**：决定了你多快能看到实验结果。

- **实时计算**（Kafka+Flink/Spark Streaming）：分钟级更新，适合快速判断"是否出bug"
- **T+1批处理**（Hive/Spark SQL）：天级更新，适合正式的统计分析（数据更稳定）

### 2.7 实验平台的数据质量管理

**为什么重要**：垃圾进，垃圾出——数据质量是所有分析的基础。

关键检查项：
- 分流日志是否有缺失（某段时间没有记录→用户分组信息丢失）
- 埋点数据是否有丢失（发版导致的采集中断）
- 异常值检测（机器人流量、测试账号、极端订单金额）
- 用户ID的稳定性（device_id是否因为用户换设备而丢失追踪）

### 2.8 实验平台的治理——权限、审核、存档

**为什么重要**：规模化的实验需要规模化的治理。

- **权限控制**：谁能创建实验？谁能修改实验参数？谁能提前停止实验？
- **审核流程**：实验上线前需要分析师审核（MDE/指标/护栏）
- **存档机制**：每个实验的完整记录（设计文档、原始数据、分析报告）存档
- **知识库**：实验结论沉淀→后来者可以搜索"这个策略有没有人测过？"</cell>


## 💻 区块3：动手练习

### 步骤1：验证Hash分流的均匀性

模拟hash分流算法，检验分流是否均匀

In [ ]:
import hashlib
import numpy as np
from scipy import stats

print("=" * 60)
print("步骤1：Hash分流的均匀性验证")
print("=" * 60)

def hash_mod(user_id, mod=100):
    """模拟hash分流：hash(user_id) % 100"""
    h = hashlib.md5(str(user_id).encode()).hexdigest()
    return int(h, 16) % mod

# 模拟100万用户的分流
n_users = 1000000
buckets = [hash_mod(i) for i in range(n_users)]
bucket_counts = np.bincount(buckets)

print(f"\n{100}个桶（bucket），{n_users:,}个用户：")
print(f"期望每桶用户数：{n_users/100:,.0f}")
print(f"实际最小值：{bucket_counts.min():,}，最大值：{bucket_counts.max():,}")
print(f"标准差：{bucket_counts.std():.0f}")

# 检验均匀性
_, p_uniform = stats.chisquare(bucket_counts)
print(f"卡方检验p值：{p_uniform:.4f} {'✓ 均匀' if p_uniform > 0.05 else '⚠ 不均匀'}")

print("\n" + "=" * 60)
print("步骤2：分层分流模拟——层内互斥、层间正交")
print("=" * 60)

np.random.seed(42)
n_users = 10000
layers = {
    "UI层": {"实验A": 0.5, "实验B": 0.5},
    "算法层": {"实验C": 0.3, "实验D": 0.3, "实验E": 0.4},
    "内容层": {"实验F": 0.5, "实验G": 0.5},
}

# 为每个用户生成每层的bucket（使用不同seed保证层间独立）
user_buckets = {}
for layer_name, experiments in layers.items():
    layer_seed = hash(layer_name) % 10000
    user_buckets[layer_name] = {
        user_id: hash_mod(f"{user_id}_{layer_seed}", 100)
        for user_id in range(n_users)
    }

# 层内互斥验证
for layer_name, experiments in layers.items():
    exp_names = list(experiments.keys())
    thresholds = {}
    cumulative = 0
    for exp_name, ratio in experiments.items():
        thresholds[exp_name] = (cumulative, cumulative + int(ratio * 100))
        cumulative += int(ratio * 100)

    # 检查每个用户是否只在一个实验中
    exp_counts = {e: 0 for e in exp_names}
    for user_id in range(n_users):
        bucket = user_buckets[layer_name][user_id]
        for exp_name, (lo, hi) in thresholds.items():
            if lo <= bucket < hi:
                exp_counts[exp_name] += 1
                break
    print(f"\n{layer_name}（层内互斥）：")
    for exp_name, expected_ratio in experiments.items():
        actual_ratio = exp_counts[exp_name] / n_users
        print(f"  {exp_name} 期望{expected_ratio:.0%}，实际{actual_ratio:.1%}")

# 层间正交验证
print("\n层间正交验证（UI层实验A vs 算法层实验C）：")
same_group = 0
for user_id in range(n_users):
    ui_bucket = user_buckets["UI层"][user_id]
    algo_bucket = user_buckets["算法层"][user_id]
    # 简单判断：都在实验组/对照组
    ui_treatment = ui_bucket < 50  # UI实验A
    algo_treatment = algo_bucket < 30  # 算法实验C
    if ui_treatment == algo_treatment:
        same_group += 1
print(f"两层都在'实验组'或都在'对照组'的比例：{same_group/n_users:.1%}")
print(f"→ 期望50%（独立分配），实际{same_group/n_users:.1%}")

print("\n" + "=" * 60)
print("步骤3：SRM检查")
print("=" * 60)

# 模拟可能的分流bug
np.random.seed(123)
# 正常分流：预期50:50
n_expected = n_users // 2
n_control = np.random.binomial(n_users, 0.5)
n_experiment = n_users - n_control

chi2 = (n_control - n_expected)**2 / n_expected + (n_experiment - n_expected)**2 / n_expected
p_srm = 1 - stats.chi2.cdf(chi2, 1)
print(f"正常分流：对照{n_control:,}，实验{n_experiment:,}，SRM p={p_srm:.4f} {'✓ 正常' if p_srm > 0.01 else '⚠ SRM异常！'}")

# 异常分流（有bug）
n_control_bug = np.random.binomial(n_users, 0.48)  # 偏离了2%
n_experiment_bug = n_users - n_control_bug
chi2_bug = (n_control_bug - n_expected)**2 / n_expected + (n_experiment_bug - n_expected)**2 / n_expected
p_srm_bug = 1 - stats.chi2.cdf(chi2_bug, 1)
print(f"异常分流：对照{n_control_bug:,}，实验{n_experiment_bug:,}，SRM p={p_srm_bug:.4f} {'✓ 正常' if p_srm_bug > 0.01 else '⚠ SRM异常！'}")

print("\n💡 SRM是实验平台的第一道防线——每次看实验结果前必须先查SRM")

## 📊 区块4：业务场景应用

### 场景1：平台的GMV和我算的不一样

**背景**：你手动算的实验组GMV提升了3.2%，平台显示提升了1.8%。差异巨大。

**排查步骤**：
1. 查数据源：你用的离线Hive表（含退款后修正），平台用的实时流（不含退款修正）→ 差异来源1
2. 查时间窗口：你统计的是"实验期间所有订单"，平台统计的是"进入实验后第2天起的订单"（有1天缓冲期）→ 差异来源2
3. 查异常值：你手动去掉了3个"异常大订单"（总金额50万），平台没有去→ 差异来源3

**结论**：三个差异来源加起来正好解释了1.4%的差距。平台的数据口径需要和业务方对齐后统一。

### 场景2：新来的产品经理不会用实验平台

**背景**：产品经理第一次用平台配置实验，设置了1%的流量、跑了2天就停了、说"p<0.05，上线吧"。

**你的检查**：
- 1%流量×2天→每组只有200人→反向MDE=30%+——只能检测巨大的效果
- p<0.05但效应量只有+0.3%——大概率是假阳性（小样本下的随机波动）
- 没查SRM——200人的分流可能不均匀

**干预**：教育产品经理正确的实验流程——先算MDE→确定样本量→跑够时间→再看p值。

### 场景3：平台宕机了——如何手动分析实验

**背景**：实验平台的数据管道挂了2天，实验报表停止更新。业务方追问"实验到底有没有效果"。

**应急方案**：
1. 从原始日志中提取分流信息（分流日志通常独立于指标计算管道）
2. 手动JOIN用户行为数据
3. 用Python做t检验和CUPED分析
4. 出临时分析报告

**教训**：分析师必须能脱离平台做分析——平台是工具，不是拐杖。</cell>


## ✅ 区块5：分级检验清单

### 🟢 初级检验
- [ ] 能说出实验平台的六大模块
- [ ] 能解释hash分流的基本原理
- [ ] 能描述分层分流的"层内互斥、层间正交"
- [ ] 能在看实验结果前检查SRM

### 🟡 中级检验
- [ ] 能分析"平台p值和手动p值不一致"的原因
- [ ] 能区分实时数据和T+1数据的使用场景
- [ ] 能解释为什么需要分层（避免多实验互相干扰）
- [ ] 能说出至少3个指标口径差异的来源

### 🔴 高级检验
- [ ] 能设计一个简化的实验平台架构（包括分流、指标、统计三大模块）
- [ ] 能评估商业实验平台vs自建平台的ROI
- [ ] 能独立排查实验平台的数据异常
- [ ] 能设计实验平台的权限和审核流程</cell>


## 🔗 区块6：跨模块连接

| 连接方向 | 关联内容 | 为什么有关 |
|---------|---------|----------|
| **← 前置依赖** | W7-W8所有实验知识 | 平台是这些知识的产品化封装 |
| **→ 后续用途** | 数据仓库与ETL | 指标计算Pipeline是数仓的核心应用 |
| **→ 后续用途** | 数据质量管理 | 平台的数据质量直接影响实验结果可信度 |
| **工程连接** | 后端分流SDK | hash算法、层域管理是后端工程的核心 |
| **工程连接** | 实时数据Pipeline | Kafka+Flink用于实时指标计算 |

### 技术栈

| 模块 | 常用技术 |
|------|---------|
| 分流服务 | 自研SDK（hash/user属性表） |
| 数据采集 | Kafka + 埋点SDK |
| 指标计算 | Spark/Flink + SQL |
| 统计分析 | Python/R + statsmodels/scipy |
| 结果展示 | BI工具（Tableau/Superset/自研） |</cell>


## 📝 区块7：当日小结

> **今天最重要的一个认知**：实验平台降低了"做实验"的门槛，但不降低"理解实验"的门槛。一个好的实验平台让你"不用自己算"，但一个合格的分析师必须能"验证平台算的结果"。理解分流算法、指标口径、统计方法——否则平台告诉你"显著"你就信了。

**五条金句带回家**：
1. 实验平台的六大模块：配置→分流→采集→指标→统计→展示
2. 分层分流：层内互斥（同一层只有一个实验）、层间正交（不同层独立分流）
3. SRM是实验的第一道防线——每次看结果前先查SRM
4. 平台p值和你手算p值不一样→先查口径差异（数据源/窗口/异常值/分母）
5. 分析师必须能脱离平台做分析——平台是工具，不是拐杖

---

*03analyst_thinking · Week 8 Day 4 · 实验平台与架构 · 2026-05-21*</cell>
